# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. `[1. 2. 3.] [1 2 3]` - оператор `/` используется для деления и результатом всегда будет число с плавающей запятой. А оператор `//` выполняет целочисленное деление с округлением вниз. Другими словами, он округляет ответ в меньшую сторону.
2. `[4. 4. 4.]` - матрица единиц суммируется по каждому строку.
3. `[[0 1 2] [1 2 3] [2 3 4]]` - получаем двумерную матрицу размером 3×3, полученную за счёт broadcasting. `[0, 1, 2]` дублируется 3 раза по вертикали. А `[[0], [1], [2]]` дублируется 3 раза по горизонтали. Затем складываются друг с другом.
4. `6.0` - `sqrt()` возвращает кавдратную корень каждого элемента массива: `[1. 2. 3.]`. Затем `sum()` суммирует все элементы.
5. `7.5` - возведением массив чисел `[1 2 3 4]` во вторую степень, получаем: `[1 4 9 16]`. Затем `mean()` находить средную значению.

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [2]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [3]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [5]:
# 1
print("Строк осталось:", clean.sum())

# Выручка за сентябрь и октябрь
revenue_sep = amount[clean & (month == "2026-09")].sum()
revenue_oct = amount[clean & (month == "2026-10")].sum()

# Изменение в %
pct_change = (revenue_oct - revenue_sep) / revenue_sep * 100

print(f"Выручка сентября: {revenue_sep:,.0f}")
print(f"Выручка октября: {revenue_oct:,.0f}")
print(f"Изменение выручки: {pct_change:.2f}%")
print("Изменение выручки не совпадает с −22,74% из ДЗ 2. Разница связана с тем, что исходное значение −22,74% также рассчитал дубликатов и тестовых записов клиента TEST.")

Строк осталось: 4739
Выручка сентября: 35,243,410
Выручка октября: 34,419,520
Изменение выручки: -2.34%
Изменение выручки не совпадает с −22,74% из ДЗ 2. Разница связана с тем, что исходное значение −22,74% также рассчитал дубликатов и тестовых записов клиента TEST.


- В октябре выручка снизилась на 2,34% относительно сентября.

## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [8]:
# 2
categories = np.unique(cat)
months = np.array(["2026-09", "2026-10"])
# считаем выручку для каждой категории отдельно по каждому месяцу
revenue_table = np.array([[amount[clean & (cat == c) & (month == m)].sum() for m in months] for c in categories])

print(f"{'Категория':<15} | {'Сентябрь':>15} | {'Октябрь':>15}")
print("-" * 50)
for c, r in zip(categories, revenue_table):
    print(f"{c:<15} | {r[0]:>15} | {r[1]:>15}")

print(f"\n{'Категория':<15} | {'Выручка':>15}")
total_by_category = revenue_table.sum(axis=1)

for c, total in zip(categories, total_by_category):
    print(f"{c:<15} | {total:>15,.0f}")
    
best_category =categories[total_by_category.argmax()]

print("Категория-лидер по выручке:", best_category, "\n")

print(f"{'Категория':<15} | {'Изменение':>13}")
cat_pct_change = ((revenue_table[:, 1] - revenue_table[:, 0]) / revenue_table[:, 0] * 100)

for c, cat_change in zip(categories, cat_pct_change):
    print(f"{c:<15} | {cat_change:>13.2f}%")
    
worst_category = categories[np.argmin(cat_pct_change)]

print("Сильнее всех упала в %:", worst_category)

Категория       |        Сентябрь |         Октябрь
--------------------------------------------------
Дом             |       4997730.0 |       5125790.0
Книги           |       1065300.0 |       1140910.0
Красота         |       2610950.0 |       2678990.0
Одежда          |       9167100.0 |       8505730.0
Электроника     |      17402330.0 |      16968100.0

Категория       |         Выручка
Дом             |      10,123,520
Книги           |       2,206,210
Красота         |       5,289,940
Одежда          |      17,672,830
Электроника     |      34,370,430
Категория-лидер по выручке: Электроника 

Категория       |     Изменение
Дом             |          2.56%
Книги           |          7.10%
Красота         |          2.61%
Одежда          |         -7.21%
Электроника     |         -2.50%
Сильнее всех упала в %: Одежда


- Самая прибыльная категория: Электроника — 34 370 430 ₸ за два месяца.
- Сильнее всех упала: Одежда — на 7,21% в октябре относительно сентября.
- Электроника приносит наибольшую выручку, а категория «Одежда» требует внимания из-за снижения продаж.

## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [9]:
# 3
count_table = np.array([[np.sum(clean & (cat == c) & (month == m)) for m in months] for c in categories])
print(f"{'Категория':<15} | {'Сентябрь':>15} | {'Октябрь':>15}")
print("Число заказов", "-" * 37)
for cat, count in zip(categories, count_table):
    print(f"{cat:<15} | {count[0]:>15} | {count[1]:>15}")

avg_check = revenue_table / count_table
print("Средний чек", "-" * 39)
for cat, avg in zip(categories, avg_check):
    print(f"{cat:<15} | {avg[0]:>15.2f} | {avg[1]:>15.2f}")

oct_check = avg_check[:, 1]
best_idx = np.argmax(oct_check)

print("Категория:", categories[best_idx])
print(f"Средний чек: {oct_check[best_idx]:,.2f} ₸")

Категория       |        Сентябрь |         Октябрь
Число заказов -------------------------------------
Дом             |             476 |             476
Книги           |             265 |             299
Красота         |             422 |             435
Одежда          |             678 |             652
Электроника     |             531 |             505
Средний чек ---------------------------------------
Дом             |        10499.43 |        10768.47
Книги           |         4020.00 |         3815.75
Красота         |         6187.09 |         6158.60
Одежда          |        13520.80 |        13045.60
Электроника     |        32772.75 |        33600.20
Категория: Электроника
Средний чек: 33,600.20 ₸


- Электроника имеет самый высокий средний чек в октябре — 33 600,20 ₸ на заказ.

## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [20]:
# 4
cities = np.unique(city)
months = np.array(["2026-09", "2026-10"])

table = np.array([
    [
        amount[clean & (city == c) & (month == m)].sum()
        for m in months
    ]
    for c in cities
])

print("Выручка каждого города:")
print(f"{'Город':<15} | {'Сентябрь':>15} | {'Октябрь':>15}")
print("-" * 50)
for c, t in zip(cities, table):
    print(f"{c:<15} | {t[0]:>15} | {t[1]:>15}")

shares = table / table.sum(axis=0)

print("Доля в выручке:")
print(f"{'Город':<15} | {'Сентябрь':>15} | {'Октябрь':>15}")
print("-" * 50)
for c, s in zip(cities, shares):
    print(f"{c:<15} | {s[0]*100:>15.2f}% | {s[1]*100:>15.2f}%")
print("Сумма долей по месяцам:", shares.sum(axis=0))

shymkent = shares[cities == "Шымкент"]

shymkent_shares = shymkent[0]

print(f"Сентябрь: {shymkent_shares[0] * 100:.2f}%")
print(f"Октябрь: {shymkent_shares[1] * 100:.2f}%")

diff = (shymkent_shares[1] - shymkent_shares[0]) * 100

print(f"Изменение: {diff:.2f} п.п.")

Выручка каждого города:
Город           |        Сентябрь |         Октябрь
--------------------------------------------------
Актобе          |       2576960.0 |       2540420.0
Алматы          |      14501070.0 |      14643020.0
Астана          |       9034220.0 |       8850530.0
Атырау          |        923990.0 |        969860.0
Караганда       |       3648520.0 |       3371680.0
Шымкент         |       4558650.0 |       4044010.0
Доля в выручке:
Город           |        Сентябрь |         Октябрь
--------------------------------------------------
Актобе          |            7.31% |            7.38%
Алматы          |           41.15% |           42.54%
Астана          |           25.63% |           25.71%
Атырау          |            2.62% |            2.82%
Караганда       |           10.35% |            9.80%
Шымкент         |           12.93% |           11.75%
Сумма долей по месяцам: [1. 1.]
Сентябрь: 12.93%
Октябрь: 11.75%
Изменение: -1.19 п.п.


- Доля Шымкента в общей выручке снизилась с 12,93% в сентябре до 11,75% в октябре.
- Изменение Шымкента: -1.19 п.п.

## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [22]:
# 5
city_change = table[:, 1] - table[:, 0]

total_change = city_change.sum()

print("Изменение по городам:", city_change)
print("Общее изменение:", total_change)

contribution = city_change / total_change * 100

print(f"{'Город':<15} | {'Изменение выручки':>15} | {'Вклад в падение':>15}")
for c, change, share in zip(cities, city_change, contribution):
    print(f"{c:<15} | {change:>15,.0f} ₸ | {share:>15.2f}%")

Изменение по городам: [ -36540.  141950. -183690.   45870. -276840. -514640.]
Общее изменение: -823890.0
Город           | Изменение выручки | Вклад в падение
Актобе          |         -36,540 ₸ |            4.44%
Алматы          |         141,950 ₸ |          -17.23%
Астана          |        -183,690 ₸ |           22.30%
Атырау          |          45,870 ₸ |           -5.57%
Караганда       |        -276,840 ₸ |           33.60%
Шымкент         |        -514,640 ₸ |           62.46%


- Основное снижение выручки приходится на Шымкент (−514 640 ₸, или 62,46% общего падения), поэтому стоит в первую очередь проверить динамику количества заказов, среднего чека и продаж по категориям в этом городе, чтобы выявить причину снижения.

## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [36]:
# 6
print(table.shape)
print(table.sum(axis=1).shape)
# Доля сентября и октября внутри каждого города
share = table / table.sum(axis=1)[:, None] # превращает массив (6,) в (6, 1)
# share = table / table.sum(axis=1, keepdims=True) - 
print(share.shape)

print(share * 100)

# Проверяем сумму долей внутри каждого города должна быть равна 1
print(share.sum(axis=1))

# Выводим результат в процентах
for i, c in enumerate(cities):
    print(
        c,
        f"Сентябрь: {share[i, 0] * 100:.2f}%",
        f"Октябрь: {share[i, 1] * 100:.2f}%"
    )

(6, 2)
(6,)
(6, 2)
[[50.35701863 49.64298137]
 [49.75646864 50.24353136]
 [50.51353807 49.48646193]
 [48.78897484 51.21102516]
 [51.9717387  48.0282613 ]
 [52.99116785 47.00883215]]
[1. 1. 1. 1. 1. 1.]
Актобе Сентябрь: 50.36% Октябрь: 49.64%
Алматы Сентябрь: 49.76% Октябрь: 50.24%
Астана Сентябрь: 50.51% Октябрь: 49.49%
Атырау Сентябрь: 48.79% Октябрь: 51.21%
Караганда Сентябрь: 51.97% Октябрь: 48.03%
Шымкент Сентябрь: 52.99% Октябрь: 47.01%


- Проблема в несовпадении размерностей массивов и не учтено правила broadcasting. table имеет размерность (6, 2) — 6 городов и 2 месяца. table.sum(axis=1) имеет размерность (6,) — сумма выручки по каждому городу.
- `[:, None]` — превращает массив (6,) в (6, 1), чтобы NumPy мог выполнить деление.
- Также использование аргумента `keepdims=True` может сохранят размерностей после суммирования, благодаря чему NumPy правильно выполняет broadcasting.

## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [38]:
# 7
# Берём только чистые оплаченные заказы
x = amount[clean]

# Рассчитываем z-оценку
z = (x - x.mean()) / x.std()

print("Среднее:", x.mean())
print("Стандартное отклонение:", x.std())

outliers = np.abs(z) > 3

outliers_count = outliers.sum()
outliers_share = outliers_count / len(z) * 100

print("Количество заказов:", len(x))
print("Количество выбросов:", outliers_count)
print(f"Доля выбросов: {outliers_share:.2f}%")

print("Среднее z:", z.mean())
print("Стандартное отклонение z:", z.std())

Среднее: 14699.921924456636
Стандартное отклонение: 13291.562590552236
Количество заказов: 4739
Количество выбросов: 87
Доля выбросов: 1.84%
Среднее z: 7.04695264417065e-17
Стандартное отклонение z: 1.0


- 87 заказов (1,84%) имеют нетипично большие или маленькие суммы относительно среднего чека; их стоит проверить на аномалии и ошибки данных.

## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [40]:
# 8
# Исключаем пропуски в days
mask = clean & ~np.isnan(days)                  # isnan(days) — находит пропущенные значения

# Создаём матрицу признаков
X = np.column_stack((amount[mask], days[mask])) # объединяет признаки в матрицу

# Масштабируем признаки, каждый столбец в диапазон [0, 1]
X_scaled = ((X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0)))

# Проверяем размерность матрицы
print("Размер X:", X.shape)
print("Размер X_scaled:", X_scaled.shape)

# Проверяем минимумы и максимумы каждого столбца
print("Min:", X_scaled.min(axis=0))
print("Max:", X_scaled.max(axis=0))

Размер X: (4595, 2)
Размер X_scaled: (4595, 2)
Min: [0. 0.]
Max: [1. 1.]


- Масштабирование нужно, чтобы признаки с большими числовыми значениями не оказывали чрезмерного влияния на обучение модели.  Приведение к одному масштабу помогает модели корректно учитывать оба признака.